# Entrenamiento final reproducible: DT e IQL

Este notebook forma parte de la entrega del TP. Documenta y reproduce el **entrenamiento de las configuraciones finales** de Decision Transformer (DT) e Implicit Q-Learning (IQL), obtenidas en la etapa de búsqueda de hiperparámetros.

- **No realiza búsqueda ni grilla**: fija las configuraciones resultantes y entrena las semillas finales.
- La búsqueda completa, la evaluación con 400 usuarios, el análisis de logs y el test están en los notebooks de búsqueda (`dt_gpu*.ipynb`, `IQL_gpu*.ipynb`) y en `comparacion.ipynb`.
- Acá se dejan **checkpoints, configuraciones e historiales reproducibles** y un chequeo rápido de validación. No reemplaza la tabla final ni usa test.

Los dos escenarios de logs del TP se cubren: `random` (logs originales uniformes) y `mixto` (logs de un sistema anterior con calidad mixta).

## 1. Protocolo de selección

La configuración se eligió **sólo con validación**, antes de tocar test:

- Partición por usuario: 90 % train / 10 % validación, estratificada por grupo. Test queda reservado.
- Episodios de 20 pasos; retorno pendiente calculado dentro del episodio.
- Métrica principal: **retorno normalizado** entre la política aleatoria y el oráculo. Secundarias: hit rate y NDCG@10.
- DT: búsqueda aleatoria de arquitectura y entrenamiento; para la configuración ganadora se comparan dos variantes de `group_dropout` (0,0 y 0,3) y se elige la primaria por validación. El retorno pedido se elige dentro del soporte de train.
- IQL: búsqueda aleatoria de arquitectura y luego grilla completa de τ, β y γ, incluyendo γ=0 frente a γ>0. La selección usa el **mínimo entre π y máximo de Q** para no sacrificar una política por la otra.
- Evaluación de la búsqueda: subconjunto estratificado de validación (50 usuarios por grupo) con 2 semillas de evaluación. La tabla final usa 400 usuarios y 3 semillas, y test se toca una sola vez.
- Las líneas base se ajustan con los **mismos logs** que el modelo. Con `mixto` se agrega el sistema anterior con ε=0 como referencia.

## 2. Espacio de búsqueda

### Decision Transformer

| Hiperparámetro | Valores explorados |
|---|---|
| `d_model` | 128, 192, 256 |
| `n_heads` | 4, 8 (compatible con `d_model`) |
| `n_layers` | 2, 3, 4 |
| `dropout` | 0,0; 0,1; 0,2 |
| `lr` | log-uniforme en [3e-4; 2e-3] |
| `weight_decay` | 1e-2; 1e-1 |
| `group_dropout` | 0,0; 0,3 |
| `batch_size` | 512, 1024, 2048 |
| `epochs` | 20, 30, 40 |
| Retorno pedido | 81, 89, 97 (dentro del soporte); 110 y 125 sólo para analizar extrapolación |

Se entrenaron 12 trials de HPO por tipo de logs. La variante primaria de `group_dropout` se eligió en validación.

### IQL

| Hiperparámetro | Valores explorados |
|---|---|
| `d_model` | 64, 96, 128 |
| `n_heads` | 4, 8 (compatible con `d_model`) |
| `n_layers` | 2, 3 |
| `dropout` | 0,0; 0,1; 0,2 |
| `lr` | log-uniforme en [1e-4; 1e-3] |
| `weight_decay` | 1e-2; 1e-1 |
| `group_dropout` | 0,3 fijo para entrenar el token "grupo desconocido" |
| `batch_size` | 256, 512, 1024 |
| `epochs` | 10, 15 |
| τ (expectil) | 0,7; 0,9 |
| β (AWR) | 3; 10 |
| γ | 0; 0,9 |

Se entrenaron 8 trials de arquitectura y una grilla de 8 configuraciones τ/β/γ sobre la arquitectura ganadora.

## 3. Configuración resultante

### DT

| Logs | `d_model` | heads | capas | dropout | lr | wd | `gd` | batch | épocas | target | score val |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| random | 128 | 4 | 3 | 0,2 | 9,66e-4 | 0,01 | 0,0 | 1024 | 40 | 97 | 0,576 |
| mixto | 128 | 8 | 3 | 0,0 | 4,56e-4 | 0,01 | 0,0 | 1024 | 20 | 97 | 0,602 |

En ambos escenarios ganó la variante `group_dropout=0.0`; la 0,3 quedó como comparación. El target 97 está en el borde superior del soporte de train (máximo 97); pedir 110/125 no mejora y se interpreta como extrapolación.

### IQL

| Logs | `d_model` | heads | capas | dropout | lr | wd | `gd` | batch | épocas | τ | β | γ | score val |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| random | 128 | 4 | 2 | 0,0 | 5,38e-4 | 0,1 | 0,3 | 512 | 10 | 0,7 | 3 | 0,9 | 0,599 |
| mixto | 128 | 4 | 2 | 0,0 | 5,38e-4 | 0,1 | 0,3 | 512 | 10 | 0,9 | 3 | 0,0 | 0,604 |

El score es el normalizado medio en validación de la configuración final. En IQL, Q tiende a preferir γ=0 y π apenas γ>0; las diferencias son chicas y se reportan en la grilla del notebook de búsqueda.

> Los valores exactos y las tablas completas están en `outputs/comparacion/` y en los notebooks de búsqueda. Acá se fija la configuración para reproducir el entrenamiento.

In [ ]:
DATASET = "netflix"
HORIZON = 20
LOGS_LIST = ("random", "mixto")
MODELS = ("dt", "iql")
SEEDS = (0, 1, 2)

MAX_BATCHES = None

RUN_EVAL = True
EVAL_PER_GROUP = 10
EVAL_SEEDS = (0,)
EVAL_BATCH_SIZE = 256

OUTPUT_ROOT = "outputs/final_local"

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "config_dataset.py").exists():
    raise RuntimeError("Ejecutar el notebook desde 2026/tp")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modelos.dt import DTPolicy
from modelos.entrenamiento import train_dt
from modelos.evaluacion import evaluate_fast
from modelos.iql import IQLPolicy, train_iql
from protocolo.datos import load_train, make_episodes, num_items, split_by_user
from protocolo.logs import logged_episodes, logs_cache
from protocolo.metricas import summarize
from protocolo.simulador import SessionConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print(f"torch {torch.__version__} | device: {DEVICE} | amp: {USE_AMP}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("CPU: el entrenamiento completo puede tardar mucho; usá MAX_BATCHES o menos semillas para probar.")

In [ ]:
SELECTED = {
    ("dt", "random"): {
        "config": dict(
            d_model=128, n_heads=4, n_layers=3, dropout=0.2,
            lr=0.0009661705275750276, weight_decay=0.01, group_dropout=0.0,
            batch_size=1024, epochs=40,
        ),
        "target_return": 97.0,
        "selection_score": 0.576,
        "selection": "HPO (12 trials); gd0.0 mejor que gd0.3 en validación; target 97 dentro del soporte",
    },
    ("dt", "mixto"): {
        "config": dict(
            d_model=128, n_heads=8, n_layers=3, dropout=0.0,
            lr=0.0004556985850036592, weight_decay=0.01, group_dropout=0.0,
            batch_size=1024, epochs=20,
        ),
        "target_return": 97.0,
        "selection_score": 0.602,
        "selection": "HPO (12 trials); gd0.0 mejor que gd0.3 en validación; target 97 dentro del soporte",
    },
    ("iql", "random"): {
        "config": dict(
            d_model=128, n_heads=4, n_layers=2, dropout=0.0,
            lr=0.0005379624172736717, weight_decay=0.1, group_dropout=0.3,
            batch_size=512, epochs=10, tau=0.7, beta=3.0, gamma=0.9,
        ),
        "target_return": None,
        "selection_score": 0.599,
        "selection": "HPO de arquitectura + grilla τ/β/γ; selección robusta min(π,Q); γ=0,9",
    },
    ("iql", "mixto"): {
        "config": dict(
            d_model=128, n_heads=4, n_layers=2, dropout=0.0,
            lr=0.0005379624172736717, weight_decay=0.1, group_dropout=0.3,
            batch_size=512, epochs=10, tau=0.9, beta=3.0, gamma=0.0,
        ),
        "target_return": None,
        "selection_score": 0.604,
        "selection": "HPO de arquitectura + grilla τ/β/γ; selección robusta min(π,Q); γ=0",
    },
}

rows = []
for (model, logs), entry in SELECTED.items():
    rows.append({
        "model": model, "logs": logs, **entry["config"],
        "target_return": entry["target_return"],
        "selection_score": entry["selection_score"],
    })
selected_table = pd.DataFrame(rows)
selected_table

In [ ]:
train_all = load_train(DATASET)
train, val = split_by_user(train_all)
N_ITEMS = num_items(DATASET)


def episodes_for(logs):
    if logs == "random":
        return make_episodes(train, HORIZON), make_episodes(val, HORIZON)
    logged_train = logged_episodes(
        logs, train, train, N_ITEMS, HORIZON, seed=0,
        cache=logs_cache(DATASET, logs, "train"),
    )
    logged_val = logged_episodes(
        logs, val, train, N_ITEMS, HORIZON, seed=1,
        cache=logs_cache(DATASET, logs, "val"),
    )
    return logged_train.episodes, logged_val.episodes


def stratified_subset(users, per_group, seed=0):
    rng = np.random.default_rng(seed)
    subset = []
    for group in range(8):
        members = [u for u in users if u.group == group]
        take = min(per_group, len(members))
        idx = rng.choice(len(members), size=take, replace=False)
        subset.extend(members[i] for i in idx)
    return subset


val_users = stratified_subset(val, EVAL_PER_GROUP, seed=0) if RUN_EVAL else []
conditions = [(0, True), (0, False), (5, True), (5, False)]
print(f"catálogo: {N_ITEMS} | usuarios de val para chequeo rápido: {len(val_users)}")

In [ ]:
out_root = Path(OUTPUT_ROOT)
out_root.mkdir(parents=True, exist_ok=True)
episodes_cache = {logs: episodes_for(logs) for logs in LOGS_LIST}

summary_rows = []
eval_rows = []

for (model_name, logs), entry in SELECTED.items():
    if logs not in LOGS_LIST or model_name not in MODELS:
        continue
    config = entry["config"]
    train_ep, val_ep = episodes_cache[logs]
    print(f"== {model_name} | logs={logs} | retorno medio de train: {train_ep.ratings.sum(1).mean():.2f}")

    if model_name == "dt":
        for seed in SEEDS:
            start = time.time()
            model, history, best_val = train_dt(
                train_ep, val_ep, N_ITEMS,
                seed=seed,
                max_batches=MAX_BATCHES,
                device=DEVICE,
                amp=USE_AMP,
                output=out_root / f"dt_{logs}" / f"seed{seed}",
                **config,
            )
            minutes = (time.time() - start) / 60
            summary_rows.append({
                "model": "dt", "logs": logs, "seed": seed,
                "selection_score": entry["selection_score"],
                "epochs": config["epochs"], "final_metric": best_val, "minutos": minutes,
            })
            print(f"DT {logs} seed {seed}: val CE {best_val:.4f} ({minutes:.1f} min)")

            if RUN_EVAL:
                policy = DTPolicy(model, target_return=entry["target_return"])
                for warmup, visible in conditions:
                    cfg = SessionConfig(horizon=HORIZON, warmup=warmup, group_visible=visible)
                    raw = evaluate_fast(
                        policy, val_users, N_ITEMS, cfg, seeds=EVAL_SEEDS,
                        batch_size=EVAL_BATCH_SIZE,
                    )
                    s = summarize(raw)
                    eval_rows.append({
                        "model": "dt", "logs": logs, "seed": seed, "mode": "dt",
                        "warmup": warmup, "group_visible": visible,
                        "normalized": float(s[("normalized", "mean")].iloc[0]),
                        "ci95": float(s[("normalized", "ci95")].iloc[0]),
                    })
            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    if model_name == "iql":
        for seed in SEEDS:
            start = time.time()
            model, history = train_iql(
                train_ep, val_ep, N_ITEMS,
                seed=seed,
                max_batches=MAX_BATCHES,
                device=DEVICE,
                amp=USE_AMP,
                output=out_root / f"iql_{logs}" / f"seed{seed}",
                **config,
            )
            minutes = (time.time() - start) / 60
            final_pi = float(history.iloc[-1]["val_pi"])
            summary_rows.append({
                "model": "iql", "logs": logs, "seed": seed,
                "selection_score": entry["selection_score"],
                "epochs": config["epochs"], "final_metric": final_pi, "minutos": minutes,
            })
            print(f"IQL {logs} seed {seed}: val π {final_pi:.4f} ({minutes:.1f} min)")

            if RUN_EVAL:
                for mode in ("pi", "q"):
                    policy = IQLPolicy(model, mode)
                    for warmup, visible in conditions:
                        cfg = SessionConfig(horizon=HORIZON, warmup=warmup, group_visible=visible)
                        raw = evaluate_fast(
                            policy, val_users, N_ITEMS, cfg, seeds=EVAL_SEEDS,
                            batch_size=EVAL_BATCH_SIZE,
                        )
                        s = summarize(raw)
                        eval_rows.append({
                            "model": "iql", "logs": logs, "seed": seed, "mode": mode,
                            "warmup": warmup, "group_visible": visible,
                            "normalized": float(s[("normalized", "mean")].iloc[0]),
                            "ci95": float(s[("normalized", "ci95")].iloc[0]),
                        })
            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

summary = pd.DataFrame(summary_rows)
summary.to_csv(out_root / "resumen_entrenamiento.csv", index=False)
display(summary.round(3))

if eval_rows:
    evaluations = pd.DataFrame(eval_rows)
    evaluations.to_csv(out_root / "validacion_rapida_sesiones.csv", index=False)
    grouped = evaluations.groupby(
        ["model", "logs", "mode", "warmup", "group_visible"]
    )["normalized"].agg(["mean", "std"]).round(3)
    grouped.to_csv(out_root / "validacion_rapida_resumen.csv")
    print("Validación rápida (media y desvío entre semillas; no es la tabla final)")
    display(grouped)

## 4. Salidas y alcance

- Checkpoints y configs: `outputs/final_local/dt_<logs>/seed<k>/` e `iql_<logs>/seed<k>/`.
- Resumen de entrenamiento: `outputs/final_local/resumen_entrenamiento.csv`.
- Chequeo rápido de validación: `validacion_rapida_resumen.csv` (pocos usuarios; **no** reemplaza la tabla final).

Para cargar un modelo:

```python
from modelos.entrenamiento import load_dt
from modelos.iql import load_iql

dt = load_dt("outputs/final_local/dt_random/seed0/modelo.pt")
iql = load_iql("outputs/final_local/iql_random/seed0/modelo_iql.pt")
```

**Alcance:** este notebook reentrena las configuraciones finales y deja evidencia reproducible. La evaluación oficial (400 usuarios, 3 semillas de evaluación, líneas base con los mismos logs y test una sola vez) se hace con los notebooks de búsqueda y `comparacion.ipynb`. En CPU el entrenamiento completo es costoso; para una prueba rápida usar `MAX_BATCHES` o `SEEDS=(0,)`.